# Cone Topology and the Origin of Flat Rotation

**Joven (2026) — builds on `dispersion_rolling.ipynb`**

## Motivation

`dispersion_rolling.ipynb` showed that flat rotation requires flat phase velocity, which requires $k \propto 1/r$ (logarithmic spiral arms). That condition was *imposed* — a constraint satisfied by adding a Lagrange multiplier (dark matter proxy).

The question here: does the geometry of the problem already provide $k \propto 1/r$ automatically, making the Lagrange multiplier unnecessary?

**Answer (to be demonstrated):** yes, if the wave equation is written on a cone rather than a flat disk. The cone's natural acoustic modes have effective Bessel order $\nu = m/n$ where $n = \sin\alpha$ is the cone parameter. The WKB condition for these modes gives $k = m/(nr) \propto 1/r$ at every radius, without constraint.

The dark matter Lagrange multiplier was computing the cost of enforcing something the topology provides for free.

## On the equilibrium zero

The density wave $\delta\Sigma$ oscillates through zero. `dispersion_rolling.ipynb` treated this as a threshold — a point where something changes character. That framing is wrong.

A violin string's zero crossings are **nodes** — positions determined by the ratio of wavelength to string length. The node at $x = L/2$ of the second harmonic doesn't mean the string's physics changes at $L/2$. It means the mode is in ratio 2:1 with the string length.

For Bessel modes on the cone: the zeros of $J_{m/n}(kr)$ are the nodes. Their positions encode the cone angle $n$ and mode number $m$. Arms and voids are regions between consecutive nodes — symmetric in $\delta\Sigma$, distinguished only by sign. The relevant quantity at any node is the **ratio** $r_{j+1}/r_j$, not the absolute value at zero.

## Ontology

Before computing: the schema of entities and relationships that structure this notebook.

### Entities

| Ket | Core properties | Role |
|-----|----------------|------|
| `\|Cone⟩` | α (half-angle), n = sin α, deficit = 2π(1−n) | geometric substrate |
| `\|Mode⟩` | m (azimuthal integer), ν = m/n (effective order), ω | natural oscillation on cone |
| `\|Geodesic⟩` | ℓ (angular momentum), E (energy) | free trajectory on cone |
| `\|DensityWave⟩` | k(r), v_φ, v_g | collective mode; v_φ flat iff k ∝ 1/r |
| `\|LogSpiral⟩` | ψ (pitch angle), k·r = const | geometric curve; ψ and α share the same parameter |
| `\|Node⟩` | r_j = j-th zero of J_{m/n} | nodal surface; ratio marker |
| `\|Arm⟩` | δΣ > 0, bounded by consecutive nodes | |
| `\|Void⟩` | δΣ < 0, bounded by consecutive nodes | |
| `\|StellarOrbit⟩` | v_c(r) | entrained to wave via stick-slip |
| `\|RotationCurve⟩` | v(r), flat: bool | observable |
| `\|Constraint⟩` | λ (Lagrange multiplier) | rolling / no-slip; λ = 0 when cone is the substrate |
| `\|DarkMatter⟩` | ρ(r) | legacy entity; reinterpreted as λ ≠ 0 artifact |

### Edges

```
|Cone⟩  ──[admits_modes]──────────────▶  |Mode⟩          # eigenfunctions J_{m/n}
|Cone⟩  ──[supports_free_motion]──────▶  |Geodesic⟩      # straight lines in unrolled sector
|Cone⟩  ──[eliminates_need_for]───────▶  |DarkMatter⟩    # central thesis

|Mode⟩  ──[WKB_implies]───────────────▶  |LogSpiral⟩     # k = m/(nr) ∝ 1/r automatically
|Mode⟩  ──[zeros_define]──────────────▶  |Node⟩          # ratio markers, not thresholds

|Node⟩  ──[bounds]────────────────────▶  |Arm⟩           # δΣ > 0 side
|Node⟩  ──[bounds]────────────────────▶  |Void⟩          # δΣ < 0 side  (symmetric)

|LogSpiral⟩  ──[instantiates]─────────▶  |DensityWave⟩
|DensityWave⟩  ──[satisfies]──────────▶  |Constraint⟩    # flat v_φ → rolling met
|DensityWave⟩  ──[entrains]───────────▶  |StellarOrbit⟩  # stick-slip locking

|Geodesic⟩  ──[projects_to]───────────▶  |RotationCurve⟩  # gives v ∝ 1/r  (wrong model)
|StellarOrbit⟩  ──[produces]──────────▶  |RotationCurve⟩  # v(r) measurement

|Constraint⟩  ──[residual_is]─────────▶  |DarkMatter⟩    # λ ≠ 0 only when cone ignored
```

The path **`|Cone⟩ → |Mode⟩ → |LogSpiral⟩ → |DensityWave⟩ → |Constraint⟩`** with λ = 0 is the claim.

The path **`|Geodesic⟩ → |RotationCurve⟩`** (v ∝ 1/r) is kept as a contrast: particles are not on free geodesics; they are entrained to the wave.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 9})

# ── Entities ──────────────────────────────────────────────────────────
ENTITIES = {
    'Cone':          {'kind': 'geometry',   'props': ['alpha', 'n=sin(alpha)', 'deficit=2pi(1-n)']},
    'Mode':          {'kind': 'wave',       'props': ['m (integer)', 'nu=m/n', 'omega']},
    'Geodesic':      {'kind': 'trajectory', 'props': ['ell', 'E']},
    'DensityWave':   {'kind': 'wave',       'props': ['k(r)', 'v_phi', 'v_g']},
    'LogSpiral':     {'kind': 'geometry',   'props': ['psi', 'k*r=const']},
    'Node':          {'kind': 'structure',  'props': ['r_j = j-th zero of J_{nu}', 'ratio r_{j+1}/r_j']},
    'Arm':           {'kind': 'structure',  'props': ['delta_Sigma > 0']},
    'Void':          {'kind': 'structure',  'props': ['delta_Sigma < 0']},
    'StellarOrbit':  {'kind': 'trajectory', 'props': ['v_c(r)']},
    'RotationCurve': {'kind': 'observable', 'props': ['v(r)', 'flat: bool']},
    'Constraint':    {'kind': 'mechanics',  'props': ['lambda (Lagrange multiplier)']},
    'DarkMatter':    {'kind': 'legacy',     'props': ['rho(r)']},
}

# ── Edges (src, dst, label, kind) ─────────────────────────────────────
EDGES = [
    ('Cone',         'Mode',          'admits_modes',         'forward'),
    ('Cone',         'Geodesic',      'supports_free_motion', 'forward'),
    ('Cone',         'DarkMatter',    'eliminates_need_for',  'negation'),
    ('Mode',         'LogSpiral',     'WKB_implies',          'forward'),
    ('Mode',         'Node',          'zeros_define',         'forward'),
    ('Node',         'Arm',           'bounds',               'forward'),
    ('Node',         'Void',          'bounds',               'forward'),
    ('LogSpiral',    'DensityWave',   'instantiates',         'forward'),
    ('DensityWave',  'Constraint',    'satisfies',            'forward'),
    ('DensityWave',  'StellarOrbit',  'entrains',             'forward'),
    ('Geodesic',     'RotationCurve', 'projects_to',          'wrong'),
    ('StellarOrbit', 'RotationCurve', 'produces',             'forward'),
    ('Constraint',   'DarkMatter',    'residual_is',          'negation'),
]

# ── Build graph ───────────────────────────────────────────────────────
G = nx.DiGraph()
for name, data in ENTITIES.items():
    G.add_node(name, **data)
for src, dst, label, kind in EDGES:
    G.add_edge(src, dst, label=label, kind=kind)

print(f"{G.number_of_nodes()} entities,  {G.number_of_edges()} edges")
print()
print("Central thesis path (lambda = 0):")
for a, b in zip(['Cone','Mode','LogSpiral','DensityWave','Constraint'],
                ['Mode','LogSpiral','DensityWave','Constraint','DarkMatter']):
    if G.has_edge(a, b):
        print(f"  |{a}⟩  [{G.edges[a,b]['label']}]  |{b}⟩")
print()
print("Contrast path (wrong model):")
print(f"  |Geodesic⟩  [projects_to]  |RotationCurve⟩   ->  v ∝ 1/r")

In [ ]:
# ── Layout: left-to-right causal flow ────────────────────────────────
POS = {
    'Cone':          (-3.0,  1.5),
    'Mode':          (-1.0,  2.5),
    'Geodesic':      (-3.0, -0.5),
    'LogSpiral':     ( 1.0,  2.5),
    'Node':          (-1.0,  0.8),
    'Arm':           ( 0.0, -0.2),
    'Void':          (-1.5, -0.8),
    'DensityWave':   ( 3.0,  2.5),
    'Constraint':    ( 3.0,  0.8),
    'StellarOrbit':  ( 3.0, -0.5),
    'RotationCurve': ( 3.0, -1.8),
    'DarkMatter':    ( 1.0,  0.8),
}

KIND_COLOR = {
    'geometry':   '#2980b9',
    'wave':       '#27ae60',
    'trajectory': '#e67e22',
    'structure':  '#8e44ad',
    'mechanics':  '#2c3e50',
    'observable': '#c0392b',
    'legacy':     '#7f8c8d',
}

EDGE_STYLE = {
    'forward':  dict(edge_color='#2c3e50', style='solid',  width=2.0),
    'negation': dict(edge_color='#c0392b', style='dashed', width=2.0),
    'wrong':    dict(edge_color='#e67e22', style='dotted', width=1.6),
}

fig, ax = plt.subplots(figsize=(15, 8))
ax.set_title('Ontology: Cone Topology → Flat Rotation', fontsize=13,
             fontweight='bold', pad=18)

for kind, style in EDGE_STYLE.items():
    edgelist = [(u, v) for u, v, d in G.edges(data=True) if d['kind'] == kind]
    if edgelist:
        nx.draw_networkx_edges(
            G, POS, edgelist=edgelist, ax=ax,
            arrowsize=20, arrowstyle='-|>',
            connectionstyle='arc3,rad=0.07',
            **style
        )

node_colors = [KIND_COLOR[ENTITIES[n]['kind']] for n in G.nodes()]
nx.draw_networkx_nodes(G, POS, ax=ax, node_color=node_colors,
                        node_size=1800, alpha=0.93)
nx.draw_networkx_labels(G, POS, ax=ax, font_size=8,
                         font_color='white', font_weight='bold')

edge_labels = {(u, v): d['label'].replace('_', '\n')
               for u, v, d in G.edges(data=True)}
nx.draw_networkx_edge_labels(
    G, POS, edge_labels, ax=ax,
    font_size=6.5, label_pos=0.40,
    bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.80)
)

# ── Legend ────────────────────────────────────────────────────────────
kind_patches = [mpatches.Patch(color=c, label=k) for k, c in KIND_COLOR.items()]
line_f = plt.Line2D([0],[0], color='#2c3e50', lw=2,          label='forward (causal)')
line_n = plt.Line2D([0],[0], color='#c0392b', lw=2, ls='--', label='negation / eliminates')
line_w = plt.Line2D([0],[0], color='#e67e22', lw=2, ls=':',  label='wrong model (contrast)')
ax.legend(handles=kind_patches + [line_f, line_n, line_w],
          loc='lower left', fontsize=8, ncol=2, framealpha=0.92)

ax.axis('off')
plt.tight_layout()
plt.show()

# ── Sanity: what does each entity connect to? ─────────────────────────
print("Degree summary:")
for node in G.nodes():
    ins  = list(G.predecessors(node))
    outs = list(G.successors(node))
    if ins or outs:
        print(f"  |{node}⟩   in: {ins}   out: {outs}")